# Neural Network Model

## Goal

Evaluate NN learning models for privacy sensitive prompt detection.

Labels:
- 0 = safe (no PII)
- 1 = privacy sensitive (contains PII)

In [1]:
import random
import numpy as np
import pandas as pd
import torch

In [2]:
SEED = 42

# Python's built-in RNG
random.seed(SEED)               

# NumPy's RNG (Global state for compatibility, or use default_rng)
np.random.seed(SEED)            

# PyTorch CPU RNG
torch.manual_seed(SEED)         

# All CUDA devices (Safe for your 4070 Ti, even if you add more GPUs later)
torch.cuda.manual_seed_all(SEED) 


## Step 1: Trust and Verify Loaded Frozen Data Splits 
Train, validation, and test sets generated by data_split.py.

In [3]:
train = pd.read_parquet('../../data_splits/train.parquet')
val = pd.read_parquet('../../data_splits/val.parquet')
test = pd.read_parquet('../../data_splits/test.parquet')

# Confirm dataset train / val / test breakdowns
summary = pd.DataFrame({
    name: {
        "rows":       len(df),
        "pct_sensitive": df["label"].mean(),                 # fraction of sensitive
        "n_blank":    df["text"].isna().sum() + (df["text"].str.strip() == "").sum(),
    }
    for name, df in [("train", train), ("val", val), ("test", test)]
}).T
print(summary)          # last line → renders as a neat 3×3 table

### CATCHES DATA CHANGES FROM OUR W1 BUILD NO FUNNY BUSINESS!!!!!
assert train.shape[0] == 260413  
assert val.shape[0] == 32552
assert test.shape[0] == 32552
###############################################


FileNotFoundError: [Errno 2] No such file or directory: '../../data_splits/train.parquet'

## STEP 2: Multilayer Preceptron (feedforward artificial neural network)
- Same as Alan's max_features = 100000 
- [100k TF-IDF features] → one linear layer → sigmoid (Alans)
- [100k TF-IDF features] → hidden layer + ReLU → output → sigmoid (Mine)

#### Compare like things for our data

**Leave the 6 blanks rows (6 out of 260,413, the impact is nil)**

## WHAT IS TfidfVectorizer
TF-IDF stands for Term Frequency-Inverse Document Frequency. It multiplies two distinct metrics:
- Term Frequency (TF): How often a word appears in a single document. (More frequent = higher score).
- Inverse Document Frequency (IDF): How common or rare a word is across your entire dataset. It penalizes universally common words like "the", "is", or "and", while boosting rare, meaningful words like "biomedical" or "scooter".

TF-IDF = Term Frequency * Inverse Document Frequency

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
# Initialize with your feature limit
vectorizer = TfidfVectorizer(max_features=100000)

# Learn from training data, then return the matrix
X_train = vectorizer.fit_transform(train["text"].fillna(""))  

# Transform validation and test data using the exact same vocabulary 
X_val   = vectorizer.transform(val["text"].fillna(""))
X_test  = vectorizer.transform(test["text"].fillna(""))


y_train = train["label"].values
y_val   = val["label"].values
y_test  = test["label"].values

#total_bytes = np.prod(X_train.shape) * 4
#print(f"{np.prod(X_train.shape) * 4 / 1e9:.1f} GB dense")  ## 

## STEP 3: PyTorch Data
- Tensor
- Dataset
- Dataloader

DataLoader — wraps a Dataset and processes data in batches, optionally shuffled. 
- Why: **Three reasons**: 
   - (1) memory — you can't hold 104 GB dense, but 256 rows densified is ~25 MB; 
   - (2) learning — gradient descent updates on mini-batches; 
   - (3) GPU throughput — GPUs love crunching a batch in parallel.

### ⚠️ DO NOT RUN THIS on your real X_train — it will try to allocate 104 GB and kill your kernel.
X_train_dense = torch.tensor(X_train.toarray(), dtype=torch.float32)   # (260413, 100000) materialized → boom   

In [ ]:
from torch.utils.data import Dataset, DataLoader

class TfidfDataset(Dataset):
    def __init__(self, X_sparse, y):
        """
        X_sparse: A SciPy sparse matrix (e.g., csr_matrix) from TfidfVectorizer
        y: A NumPy array, Pandas Series, or list of targets/labels
        """
        # Ensure the matrix is in Compressed Sparse Row (CSR) format for fast row slicing
        self.X_sparse = X_sparse.tocsr()
        
        # Convert labels to a PyTorch tensor (use torch.float32 for regression/binary, long for multi-class)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self):
        return self.X_sparse.shape[0]

    def __getitem__(self, idx):
        # 1. Slice a single row out of the sparse matrix
        row_sparse = self.X_sparse[idx]
        
        # 2. ⚠️ Convert ONLY this single row to a dense NumPy array, then to a PyTorch float tensor
        row_dense = torch.tensor(row_sparse.toarray()[0], dtype=torch.float32)
        
        # 3. Retrieve the target label
        label = self.y[idx]
        
        return row_dense, label
    
## 4 Threated DataLoaded To Use The GPU more    
train_loader = DataLoader(TfidfDataset(X_train, y_train), batch_size=256, shuffle=True,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(TfidfDataset(X_val, y_val),     batch_size=256, shuffle=False,
                          num_workers=4, pin_memory=True)

for batch_X, batch_y in train_loader:
    print(batch_X.shape, batch_y.shape)
    break

## STEP 4: BEAT Logistic Regression By Adding a hidden layer and non-linearity
```Alan Logistic Regression:   [100k features] ───────────────► [1 logit] → sigmoid```
```
Fil MLP:              [100k features] → Linear → ReLU → [1 logit] → sigmoid
                                          └── the hidden layer ──┘
```                                          
> ### 🧠 Why the non-linearity is *the whole point*
>
> We add exactly **two** things to logistic regression: a **hidden layer** and a **non-linearity**. 
> If you stack two `Linear` layers with **nothing** between them, the math *collapses*:
> $$W_2(W_1 x + b_1) + b_2 = (W_2 W_1)\,x + (W_2 b_1 + b_2) = \text{a single linear layer}$$
> So two `Linear` layers back-to-back are algebraically identical to **one** linear layer — i.e. **logistic regression with extra steps.**
>
> The **ReLU** (`max(0, x)`) between them breaks that collapse. It lets the network bend its decision boundary and learn **feature interactions** — e.g. *"token A **and** token B together signal PII"* — which a purely linear model fundamentally cannot represent.

> Without the ReLU in the middle out models fc1 in_feat = 1000000 out=256 and fc2 in_feat = 256 out_feat = 1 would collapse to 1000000x1 == logistic regression

In [ ]:
import torch.nn as nn
# MLP (Multilayer Perceptron) Class
class MLP(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()                      # MUST be first
      
        self.fc1 = nn.Linear(input_dim , hidden_dim ) # input → hidden

        self.relu = nn.ReLU()
    
        self.fc2 = nn.Linear(hidden_dim, 1)  #  hidden → 1 logit

    def forward(self, x):
        x = x.squeeze(1)  # Cleans shape from (batch_size, 1, 100000) to (batch_size, 100000)
        out = self.fc1(x)
        out = self.relu(out)
        out = self.fc2(out) # Outputs shape: (batch_size, 1)
        return out
    
model = MLP(input_dim=100000, hidden_dim=256)
print(model)                       # prints your architecture
xb, yb = next(iter(train_loader))
print(model(xb).shape)  

## STEP 5: Setup Loss, Optimizer and prep for training loop GPU RECOMMENDED

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = MLP(input_dim=100000, hidden_dim=256).to(device)   # USE THE GPU
criterion = nn.BCEWithLogitsLoss() # Takes raw logits and applies sigmoid + binary cross-entropy

optimizer = torch.optim.Adam(model.parameters(), 1e-3) #gradient-descent engine


In [ ]:
from sklearn.metrics import f1_score, recall_score
import torch

EPOCHS, PATIENCE, CKPT = 15, 3, "best_mlp.pt"   # EPOCHS is just an upper bound
best_metric, since_improve = -1.0, 0             # F1 is MAXIMIZED, so start low (not +inf)

def evaluate(model, loader, threshold=0.5):
    model.eval()
    logits_all, labels_all = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            logits_all.append(model(xb).squeeze(1).cpu())
            labels_all.append(yb)
    logits = torch.cat(logits_all)
    y_true = torch.cat(labels_all).numpy()
    y_pred = (torch.sigmoid(logits) >= threshold).int().numpy()
    return y_true, y_pred

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)     # move batch to device
        logits = model(xb).squeeze(1)             # forward → (batch,)
        loss   = criterion(logits, yb)

        optimizer.zero_grad()                     # clear old gradients
        loss.backward()                           # backprop
        optimizer.step()                          # update weights

        running_loss += loss.item()               # .item() to avoid memory ballooning

    avg_loss = running_loss / len(train_loader)

    # ---- validate ----
    yv, pv = evaluate(model, val_loader)          # re-evaluate current weights
    val_metric = f1_score(yv, pv, pos_label=1)
    print(f"Epoch {epoch+1:02d} | train_loss {avg_loss:.4f} | val_F1 {val_metric:.3f}")

    # ---- early-stopping bookkeeping if three consecutive epochs are not decreasing avg loss then stop early ----
    if val_metric > best_metric:
        best_metric, since_improve = val_metric, 0
        torch.save(model.state_dict(), CKPT)      # checkpoint the BEST weights
        print("   ↳ new best, saved")
    else:
        since_improve += 1
        print(f"   (no improve {since_improve}/{PATIENCE})")
        if since_improve >= PATIENCE:
            print(f"Early stop @ epoch {epoch+1}. Best val_F1 {best_metric:.3f}")
            break

# ---- restore the BEST model, not the degraded final one ----
state_dict = torch.load(CKPT, map_location=device, weights_only=True)
model.load_state_dict(state_dict)
model = model.to(device)

print(f"Restored best checkpoint (val_F1 {best_metric:.3f})")

## STEP 6: Validation, metrics & the threshold
### Precision vs Recall Analysis
> Graph the Precision vs Recall to find the optimal threshold we want to identify PII
> - Recall — Of every prompt that truly contained PII, what fraction did the model flag? → TP / (TP + FN)
> - Precision - Of every prompt the model flagged as PII, what fraction actually were? → TP / (TP + FP) 
> - F1 — one number that says "good at both" The harmonic mean of precision and recall: 2·P·R / (P + R) 
>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import precision_score, recall_score, f1_score, classification_report, confusion_matrix

# Don't need multiple workers on eval keeps from throwing errors
val_loader  = DataLoader(TfidfDataset(X_val,  y_val),  batch_size=256, shuffle=False, num_workers=0)
test_loader = DataLoader(TfidfDataset(X_test, y_test), batch_size=256, shuffle=False, num_workers=0)

## Evaluation for 0.5 Threshold To Compare to Classical Models
yv, pv = evaluate(model, val_loader, 0.5)
print(classification_report(yv, pv, target_names=["Safe","PII"], digits=3))
print(confusion_matrix(yv, pv))

# val probabilities, ONE forward pass (reused for all thresholds)
def get_probs(model, loader):
    model.eval()
    logits, labels = [], []
    with torch.no_grad():
        for xb, yb in loader:
            logits.append(model(xb.to(device)).squeeze(1).cpu())
            labels.append(yb)
    return torch.cat(labels).numpy(), torch.sigmoid(torch.cat(logits)).numpy()

y_true, probs = get_probs(model, val_loader)


# sweep a grid from .05 -> .95 every 0.5 save them for the graph and analysis
thresholds = np.arange(0.05, 0.96, 0.05)
prec, rec, f1s, missed = [], [], [], []
for t in thresholds:
    y_pred = (probs >= t).astype(int)
    prec.append(precision_score(y_true, y_pred, pos_label=1, zero_division=0))      
    rec.append(  recall_score(y_true, y_pred, pos_label=1) )     
    f1s.append(  f1_score(y_true, y_pred, pos_label=1))   
    missed.append(((y_true == 1) & (y_pred == 0)).sum())   # false negatives = PII let through

### Find the best f1 and optimal recal threshold for given TARGET
best_f1_t = thresholds[np.argmax(f1s)]                       
# security-oriented: highest threshold that still hits a recall floor (keeps precision as high as possible)
TARGET = 0.95
ok = [t for t, r in zip(thresholds, rec) if r >= TARGET]
recall_t = max(ok) if ok else thresholds[0]
print(f"max-F1 threshold: {best_f1_t:.2f}   |   ≥{TARGET:.0%}-recall threshold: {recall_t:.2f}")

# 3) plot
plt.figure(figsize=(8,5))
plt.plot(thresholds, prec, label="Precision (PII)")
plt.plot(thresholds, rec,  label="Recall (PII)")
plt.plot(thresholds, f1s,  label="F1 (PII)")
plt.axvline(0.5, ls="--", color="gray", label="default 0.5")
plt.axvline(best_f1_t, ls="--", color="purple", label=f"best f1 {best_f1_t:.2f}")
plt.axvline(recall_t, ls="--", color="red", label=f"recall_t {recall_t:.2f}")
plt.xlabel("Decision threshold"); plt.ylabel("Score")
plt.title("Validation threshold sweep — PII (class 1)")
plt.legend(); plt.grid(alpha=0.3); plt.show()

# STEP 7: THE TRUTH RUN THIS ONLY ONCE
## Final test evaluation 

In [ ]:
# from sklearn.metrics import classification_report, confusion_matrix
# from torch.utils.data import DataLoader

# # single pass → no workers, no shuffle
# test_loader = DataLoader(TfidfDataset(X_test, y_test), batch_size=256, shuffle=False, num_workers=0)

# # `model` must be your RESTORED best checkpoint — do NOT retrain after this
# yt, pt = evaluate(model, test_loader)        # default 0.5 → apples-to-apples with LR
# print(classification_report(yt, pt, target_names=["Safe","PII"], digits=3))
# print(confusion_matrix(yt, pt))

             precision    recall  f1-score   support

        Safe      0.685     0.681     0.683     10626
         PII      0.846     0.848     0.847     21926

    accuracy                          0.793     32552
   macro avg      0.765     0.764     0.765     32552
weighted avg      0.793     0.793     0.793     32552

[[ 7235  3391]
 [ 3331 18595]]

## STEP 8: ERROR ANALYSIS
PII the model called Safe  ← the dangerous misses
Safe the model flagged as PII

Look it over with your eyes

In [ ]:
yv, pv = evaluate(model, val_loader)
res = val.copy()
res["true"], res["pred"] = yv, pv

fn = res[(res.true == 1) & (res.pred == 0)]   # PII the model called Safe  ← the dangerous misses
fp = res[(res.true == 0) & (res.pred == 1)]   # Safe the model flagged as PII
print(f"FN={len(fn)}  FP={len(fp)}")

for t in fn["text"].head(10):
    print(repr(t[:200]))